# Run voteframe from Python

This notebook reads CSV names from `python_test/data`, detects whether the files match the initial three-file workflow or the secondary two-file workflow, runs the corresponding R CLI wrapper, and for the initial workflow builds predicted-vs-actual diagnostics using the municipality quartiles output and the election results file.

In [41]:
from pathlib import Path
import csv
import subprocess
from pprint import pprint

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import math
import warnings
from matplotlib.lines import Line2D


class _StatsFallback:
    @staticmethod
    def spearmanr(a, b):
        a_series = pd.Series(a)
        b_series = pd.Series(b)
        valid = ~(a_series.isna() | b_series.isna())
        if valid.sum() < 2:
            return np.nan, np.nan
        corr = a_series[valid].corr(b_series[valid], method="spearman")
        return float(corr), np.nan


stats = _StatsFallback()
print("Info: using pandas-based Spearman fallback (p-values set to NaN).")


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "scripts" / "run_post_strat_cli.R").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing scripts/")


def classify_csv(path: Path) -> str:
    name = path.stem.lower()
    if "survey" in name:
        return "survey"
    if "frame" in name:
        return "frame"
    if "total" in name or "totals" in name:
        return "municipality_vote_totals"
    if "vote" in name and "municip" in name:
        return "municipality_vote_totals"
    if "election" in name or "result" in name:
        return "municipality_vote_totals"
    return "unknown"


def read_header(path: Path) -> list[str]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        return next(csv.reader(handle))


def preview_rows(path: Path, n: int = 3) -> list[dict[str, str]]:
    with path.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.DictReader(handle)
        rows = []
        for i, row in enumerate(reader):
            rows.append(row)
            if i + 1 >= n:
                break
        return rows


def has_columns(header: list[str], required: set[str]) -> bool:
    return required.issubset(set(header))


repo_root = find_repo_root(Path.cwd())
data_dir = repo_root / "python_test" / "data"
output_dir = repo_root / "python_test" / "output"
secondary_runner = repo_root / "scripts" / "run_post_strat_cli.R"
initial_runner = repo_root / "scripts" / "run_initial_extension_cli.R"

n_sims = 250
year = 2022

print("Repo root:", repo_root)
print("Data dir:", data_dir)
print("Output dir:", output_dir)

Info: using pandas-based Spearman fallback (p-values set to NaN).
Repo root: /Users/danjonaitis/Documents/GitHub/voteframe
Data dir: /Users/danjonaitis/Documents/GitHub/voteframe/python_test/data
Output dir: /Users/danjonaitis/Documents/GitHub/voteframe/python_test/output


In [42]:
csv_files = sorted(data_dir.glob("*.csv"))
if not csv_files:
    raise ValueError("No CSV files found in python_test/data.")

classified = {
    "survey": [],
    "frame": [],
    "municipality_vote_totals": [],
    "unknown": [],
}

for path in csv_files:
    classified[classify_csv(path)].append(path)

print("CSV files found:")
for key, paths in classified.items():
    for path in paths:
        print(f"- {key}: {path.name}")

CSV files found:
- survey: survey_DNES.csv
- frame: frame.csv
- municipality_vote_totals: election_results.csv


In [43]:
survey_path = classified["survey"][0] if classified["survey"] else None
frame_path = classified["frame"][0] if classified["frame"] else None
totals_path = classified["municipality_vote_totals"][0] if classified["municipality_vote_totals"] else None

if survey_path is None:
    raise ValueError("Could not identify a survey CSV. Include 'survey' in the filename.")
if frame_path is None:
    raise ValueError("Could not identify a frame CSV. Include 'frame' in the filename.")

survey_header = read_header(survey_path)
frame_header = read_header(frame_path)
totals_header = read_header(totals_path) if totals_path else []

print("Survey file:", survey_path.name)
print("Frame file:", frame_path.name)
print("Municipality vote totals file:", totals_path.name if totals_path else "not found")
print()
print("Survey columns:")
print(survey_header)
print()
print("Frame columns:")
print(frame_header)
if totals_path:
    print()
    print("Municipality totals columns:")
    print(totals_header)

Survey file: survey_DNES.csv
Frame file: frame.csv
Municipality vote totals file: election_results.csv

Survey columns:
['Unnamed: 0.1', 'Unnamed: 0', 'birthyear', 'age', 'municipality', 'mun_code', 'number_of_children', 'Q91', 'gender', 'marital_status', 'living_situation', 'interest_in_politics', 'days_per_week_tv_news', 'days_per_week_newspaper', 'turnout_in_2022', 'past_vote', 'vote_in_2019', 'party_attachment_strength', 'class_differences_perception', 'gross_annual_income', 'economic_security', 'social_class_identity', 'union_membership', 'employment_sector', 'school_education', 'higher_education', 'education_level', 'vote_pct', 'vote_pct', 'age_group']

Frame columns:
['', 'municipality', 'gender', 'age_group', 'education', 'N']

Municipality totals columns:
['party', 'municipality', 'year', 'pop_pct']


In [44]:
secondary_survey_cols = {"age_group", "gender", "municipality", "education_level", "predicted_vote"}
secondary_frame_cols = {"age_group", "gender", "municipality", "education_level", "expected_N_raked"}

initial_survey_cols = {"age_group", "gender", "municipality", "education_level", "past_vote"}
initial_frame_cols = {"municipality", "gender", "age_group", "N"}
initial_totals_cols = {"party", "municipality"}

looks_like_secondary = has_columns(survey_header, secondary_survey_cols) and has_columns(frame_header, secondary_frame_cols)
looks_like_initial = (
    has_columns(survey_header, initial_survey_cols)
    and has_columns(frame_header, initial_frame_cols)
    and totals_path is not None
    and has_columns(totals_header, initial_totals_cols)
)

mode = "unknown"
if looks_like_secondary:
    mode = "secondary"
elif looks_like_initial:
    mode = "initial"

print("Detected mode:", mode)

Detected mode: initial


In [45]:
print("Survey preview:")
pprint(preview_rows(survey_path))
print()
print("Frame preview:")
pprint(preview_rows(frame_path))

if totals_path:
    print()
    print("Municipality vote totals preview:")
    pprint(preview_rows(totals_path))

Survey preview:
[{'Q91': '',
  'Unnamed: 0': '0',
  'Unnamed: 0.1': '0',
  'age': '33.0',
  'age_group': '30-34',
  'birthyear': '1993.0',
  'class_differences_perception': '',
  'days_per_week_newspaper': '0.0',
  'days_per_week_tv_news': '1.0',
  'economic_security': '',
  'education_level': 'Unknown',
  'employment_sector': '',
  'gender': 'Women',
  'gross_annual_income': '',
  'higher_education': '',
  'interest_in_politics': 'Not at all interested',
  'living_situation': 'Living alone',
  'marital_status': 'Single',
  'mun_code': '',
  'municipality': '',
  'number_of_children': '2.0',
  'party_attachment_strength': '',
  'past_vote': 'Did not vote',
  'school_education': '',
  'social_class_identity': '',
  'turnout_in_2022': '',
  'union_membership': '',
  'vote_in_2019': 'Did not vote',
  'vote_pct': ''},
 {'Q91': '4.0',
  'Unnamed: 0': '1',
  'Unnamed: 0.1': '1',
  'age': '93.0',
  'age_group': '90-94',
  'birthyear': '1933.0',
  'class_differences_perception': '',
  'days_pe

In [46]:
if mode == "unknown":
    raise ValueError(
        "Could not match these files to the implemented initial or secondary schemas. Check the printed headers above."
    )

output_dir.mkdir(parents=True, exist_ok=True)

if mode == "initial":
    cmd = [
        "Rscript",
        str(initial_runner),
        str(survey_path),
        str(frame_path),
        str(totals_path),
        str(output_dir),
    ]
    if year is not None:
        cmd.append(str(year))
else:
    cmd = [
        "Rscript",
        str(secondary_runner),
        str(survey_path),
        str(frame_path),
        str(output_dir),
        str(n_sims),
    ]

print("Running command:")
print(" ".join(cmd))

result = subprocess.run(cmd, cwd=repo_root, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f"R command failed with exit code {result.returncode}")

print("Finished. Outputs written to", output_dir)

Running command:
Rscript /Users/danjonaitis/Documents/GitHub/voteframe/scripts/run_initial_extension_cli.R /Users/danjonaitis/Documents/GitHub/voteframe/python_test/data/survey_DNES.csv /Users/danjonaitis/Documents/GitHub/voteframe/python_test/data/frame.csv /Users/danjonaitis/Documents/GitHub/voteframe/python_test/data/election_results.csv /Users/danjonaitis/Documents/GitHub/voteframe/python_test/output 2022


Attaching package: ‘dplyr’

The following objects are masked from ‘package:stats’:

    filter, lag

The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union

Warning messages:
1: package ‘vglmer’ was built under R version 4.3.3 
2: In check_dep_version() : ABI version mismatch: 
lme4 was built with Matrix ABI version 1
Current Matrix ABI version is 0
Please re-install lme4 from source or restore original ‘Matrix’ package
New names:
• `vote_pct` -> `vote_pct...28`
• `vote_pct` -> `vote_pct...29`
New names:
• `` -> `...1`
There are 4 municipa